# Notebook 0 — Data at a Glance

Structural orientation for the NJ ACS 5-year (2020–2024) pull before any uncertainty math. We load all three geography levels, inspect shapes and missingness, and render quick boundary maps.

**ACS (American Community Survey):** ongoing Census Bureau survey; every estimate ships with a **MOE (margin of error)** at 90% confidence.

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'JL_Analysis':
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'analysis' / 'JL_Analysis'
sys.path.insert(0, str(NOTEBOOK_DIR))

from helpers import (
    GEO_LABELS,
    GEO_LEVELS,
    VARIABLES,
    annotation_mask,
    estimate_moe_cols,
    load_acs,
    load_geo,
)

plt.rcParams['figure.dpi'] = 110

In [ ]:
acs = {level: load_acs(level) for level in GEO_LEVELS}
geo = {level: load_geo(level) for level in GEO_LEVELS}

for level in GEO_LEVELS:
    df = acs[level]
    print(f"\n{'=' * 60}")
    print(f"{GEO_LABELS[level]} — {level}")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print('\nColumns:')
    print(df.columns.tolist())
    print('\nDtypes:')
    print(df.dtypes)
    display(df.head(3))

## Row counts by geography level

Smaller geographies nest inside larger ones, so row counts explode as we zoom in — the central empirical setup for this project.

In [ ]:
counts = pd.DataFrame({
    'Geography level': [GEO_LABELS[l] for l in GEO_LEVELS],
    'Rows': [len(acs[l]) for l in GEO_LEVELS],
})
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(counts['Geography level'], counts['Rows'], color=['#2c5f8a', '#4a8fb8', '#7eb8d8'])
ax.set_title('NJ ACS rows by geography level (vintage 2024)')
ax.set_ylabel('Number of rows')
for bar, val in zip(bars, counts['Rows']):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 80, f'{val:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## Missing values and annotation codes

**Annotation codes** are giant negative sentinel values the Census API returns instead of real numbers (e.g. `-555555555` = controlled estimate, `-666666666` = insufficient sample). Our client library converts them to blanks, so this chart shows combined null rates. See `docs/glossary.md`.

In [ ]:
null_rows = []
for level in GEO_LEVELS:
    df = acs[level]
    for var_code in VARIABLES:
        est_col, moe_col = estimate_moe_cols(var_code)
        for col, kind in [(est_col, 'estimate'), (moe_col, 'MOE')]:
            if col not in df.columns:
                continue
            s = pd.to_numeric(df[col], errors='coerce')
            null_rows.append({
                'Geography level': GEO_LABELS[level],
                'Variable': VARIABLES[var_code],
                'Column type': kind,
                'Null rate': s.isna().mean(),
                'Annotation rate': annotation_mask(s).mean(),
            })

null_df = pd.DataFrame(null_rows)
pivot = null_df.pivot_table(
    index='Variable', columns='Geography level', values='Null rate', aggfunc='max'
)
fig, ax = plt.subplots(figsize=(9, 5))
pivot.plot(kind='barh', ax=ax, color=['#2c5f8a', '#4a8fb8', '#7eb8d8'])
ax.set_title('Null rate by variable and geography level')
ax.set_xlabel('Share of rows that are blank')
ax.legend(title='')
plt.tight_layout()
plt.show()

print('Block-group poverty and subgroup variables are 100% blank — expected; '
      'those tables are not published below tract level.')

## Quick boundary maps — does this look like New Jersey?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 7))
colors = ['#2c5f8a', '#4a8fb8', '#7eb8d8']
for ax, level, color in zip(axes, GEO_LEVELS, colors):
    gdf = geo[level]
    gdf.plot(ax=ax, color=color, edgecolor='white', linewidth=0.2)
    ax.set_title(f"{GEO_LABELS[level]}\n({len(gdf):,} shapes)")
    ax.set_axis_off()
fig.suptitle('New Jersey boundaries by geography level (ACS 2024 vintage)', y=1.02)
plt.tight_layout()
plt.show()